<a href="https://colab.research.google.com/github/Dacon-contest/dacon-data-segmentation_and_train/blob/main/5_fold_v4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import cv2
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler, autocast
from pathlib import Path
import timm
import albumentations as A
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
from tqdm import tqdm
from albumentations.pytorch import ToTensorV2

In [ ]:
# 2. 경로 설정 (본인의 zip 파일 경로로 수정하세요)
zip_path = "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/dacon/no_seg&seg_data.zip"
target_path = "/content/no_seg&seg_data"

# 3. /content 폴더로 복사 (드라이브에서 직접 푸는 것보다 복사 후 푸는 게 빠름)
!cp "{zip_path}" /content/

# 4. 압축 해제 (-q 옵션으로 로그 출력 방지)
zip_name = os.path.basename(zip_path)
!unzip -q "/content/{zip_name}" -d "{target_path}"

print(f"✅ {target_path}에 압축 해제 완료!")

✅ /content/no_seg&seg_data에 압축 해제 완료!


In [ ]:
# ==========================================
# 1. 설정값 (A100 최적화 및 경로)
# ==========================================
CFG = {
    'seed': 42,
    'img_size': 384,
    'batch_size': 32, # A100 기준
    'epochs': 40,
    'n_folds': 5,
    'model_name': 'convnext_small.fb_in22k_ft_in1k_384', # small 버전이 과적합 방어에 유리
    'orig_root': Path("/content/no_seg&seg_data/no_seg_data"),
    'mask_root': Path("/content/no_seg&seg_data/seg_data"),
    'train_csv': '/content/no_seg&seg_data/no_seg_data/train.csv',
    'dev_csv': '/content/no_seg&seg_data/no_seg_data/dev.csv',
    'test_csv': '/content/no_seg&seg_data/no_seg_data/sample_submission.csv'
}

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(CFG['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


In [ ]:
train_df = pd.read_csv(CFG['train_csv'])
dev_df = pd.read_csv(CFG['dev_csv'])

train_df['folder'] = 'train'
dev_df['folder'] = 'dev'

all_df = pd.concat([train_df, dev_df], axis=0).reset_index(drop=True)
# 🚨 unstable을 1로 통일! (나중에 제출할 때 헷갈리지 않기 위함)
all_df['label_int'] = all_df['label'].map({'unstable': 1, 'stable': 0})
all_df['stratify_key'] = all_df['label'].astype(str) + "_" + all_df['folder']

In [ ]:
train_transform = A.Compose([
    A.Resize(CFG['img_size'], CFG['img_size']), # IMG_SIZE 대신 CFG 사용
    A.HorizontalFlip(p=0.5),
    A.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1, p=0.8),
    A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, border_mode=0, p=0.6),
    A.GaussianBlur(blur_limit=(3, 7), p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(), # 이제 에러 안 남
])

val_transform = A.Compose([
    A.Resize(CFG['img_size'], CFG['img_size']),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # 검증용 Normalize도 추가 추천
    ToTensorV2(),
])

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [ ]:
class StabilityDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self): return len(self.df)

    def _load_img_4ch(self, img_id, view, folder):
        orig_path = CFG['orig_root'] / folder / view / f"{img_id}.png"
        mask_path = CFG['mask_root'] / folder / view / f"{img_id}.png"

        # RGB 이미지
        orig_img = cv2.imread(str(orig_path))
        orig_img = cv2.cvtColor(orig_img, cv2.COLOR_BGR2RGB)

        # Mask 이미지 (흑백 1채널)
        mask_img = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

        return orig_img, mask_img

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id, folder = row['id'], row['folder']

        f_img, f_mask = self._load_img_4ch(img_id, 'front', folder)
        t_img, t_mask = self._load_img_4ch(img_id, 'top', folder)

        if self.transform:
            # 이미지와 마스크를 동시에 넘겨야 증강이 똑같이 적용됨 (ColorJitter는 이미지에만 적용됨)
            f_res = self.transform(image=f_img, mask=f_mask)
            t_res = self.transform(image=t_img, mask=t_mask)

            f_img = f_res['image']
            f_mask = f_res['mask'].unsqueeze(0).float() / 255.0 # [1, H, W]

            t_img = t_res['image']
            t_mask = t_res['mask'].unsqueeze(0).float() / 255.0

        # 3채널 + 1채널 = 4채널
        f_input = torch.cat([f_img, f_mask], dim=0)
        t_input = torch.cat([t_img, t_mask], dim=0)

        return f_input, t_input, torch.tensor(row['label_int'], dtype=torch.float32)

In [ ]:
class UltimateFusionNet(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        # 4채널 입력 (in_chans=4)
        self.backbone = timm.create_model(model_name, pretrained=True, in_chans=4, num_classes=0)
        feat_dim = self.backbone.num_features

        self.head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(feat_dim * 2, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, 1),
        )

    def forward(self, front, top):
        f = self.backbone(front)
        t = self.backbone(top)
        return self.head(torch.cat([f, t], dim=1)).squeeze(-1)

In [ ]:
skf = StratifiedKFold(n_splits=CFG['n_folds'], shuffle=True, random_state=CFG['seed'])

for fold, (t_idx, v_idx) in enumerate(skf.split(all_df, all_df['stratify_key'])):
    print(f"\n🚀 Fold {fold+1} 시작 (4채널 증강 완벽 적용)")
    train_fold, val_fold = all_df.iloc[t_idx], all_df.iloc[v_idx]

    train_ds = StabilityDataset(train_fold, transform=train_transform)
    val_ds = StabilityDataset(val_fold, transform=val_transform)

    train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, num_workers=4)
    val_loader = DataLoader(val_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=4)

    model = UltimateFusionNet(CFG['model_name']).to(device)
    optimizer = torch.optim.AdamW([
        {'params': model.backbone.parameters(), 'lr': 3e-5},
        {'params': model.head.parameters(), 'lr': 3e-4},
    ], weight_decay=0.01)

    criterion = nn.BCEWithLogitsLoss()
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG['epochs'])
    scaler = GradScaler('cuda')

    best_val_logloss = float('inf')

    for epoch in range(CFG['epochs']):
        model.train()
        train_loss = 0
        for f, t, l in tqdm(train_loader, desc=f"Fold {fold+1} Ep {epoch+1}"):
            f, t, l = f.to(device), t.to(device), l.to(device)
            optimizer.zero_grad()
            with torch.amp.autocast('cuda'):
                logits = model(f, t)
                loss = criterion(logits, l)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()

        model.eval()
        val_probs, val_labels = [], []
        with torch.no_grad():
            for f, t, l in val_loader:
                f, t = f.to(device), t.to(device)
                logits = model(f, t)
                probs = torch.sigmoid(logits)
                val_probs.extend(probs.cpu().numpy())
                val_labels.extend(l.numpy())

        epoch_logloss = log_loss(val_labels, val_probs, labels=[0, 1])
        scheduler.step()

        print(f"Fold {fold+1} | Ep {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Val LogLoss: {epoch_logloss:.6f}")

        if epoch_logloss < best_val_logloss:
            best_val_logloss = epoch_logloss
            torch.save(model.state_dict(), f'best_model_fold{fold+1}.pth')
            print(f"🌟 Fold {fold+1} 최고기록! (LogLoss: {epoch_logloss:.6f})")


🚀 Fold 1 시작 (4채널 증강 완벽 적용)


Fold 1 Ep 1: 100%|██████████| 28/28 [00:21<00:00,  1.28it/s]


Fold 1 | Ep 1 | Train Loss: 0.3187 | Val LogLoss: 0.051009
🌟 Fold 1 최고기록! (LogLoss: 0.051009)


Fold 1 Ep 2: 100%|██████████| 28/28 [00:09<00:00,  2.97it/s]


Fold 1 | Ep 2 | Train Loss: 0.0538 | Val LogLoss: 0.019652
🌟 Fold 1 최고기록! (LogLoss: 0.019652)


...

Fold 5 | Ep 38 | Train Loss: 0.0015 | Val LogLoss: 0.000713


Fold 5 Ep 39: 100%|██████████| 28/28 [00:09<00:00,  2.98it/s]


Fold 5 | Ep 39 | Train Loss: 0.0054 | Val LogLoss: 0.000681


Fold 5 Ep 40: 100%|██████████| 28/28 [00:09<00:00,  2.97it/s]


Fold 5 | Ep 40 | Train Loss: 0.0013 | Val LogLoss: 0.000729


In [ ]:
class StabilityTestDataset(Dataset):
    def __init__(self, df, transform=None):
        self.df = df
        self.transform = transform

    def __len__(self): return len(self.df)

    def _load_img_4ch(self, img_id, view):
        orig_path = CFG['orig_root'] / 'test' / view / f"{img_id}.png"
        mask_path = CFG['mask_root'] / 'test' / view / f"{img_id}.png"

        orig_img = cv2.cvtColor(cv2.imread(str(orig_path)), cv2.COLOR_BGR2RGB)
        mask_img = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        return orig_img, mask_img

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['id']
        f_img, f_mask = self._load_img_4ch(img_id, 'front')
        t_img, t_mask = self._load_img_4ch(img_id, 'top')

        if self.transform:
            f_res = self.transform(image=f_img, mask=f_mask)
            t_res = self.transform(image=t_img, mask=t_mask)

            f_img = f_res['image']
            f_mask = f_res['mask'].unsqueeze(0).float() / 255.0
            t_img = t_res['image']
            t_mask = t_res['mask'].unsqueeze(0).float() / 255.0

        f_input = torch.cat([f_img, f_mask], dim=0)
        t_input = torch.cat([t_img, t_mask], dim=0)

        return f_input, t_input

submit = pd.read_csv(CFG['test_csv'])
test_ds = StabilityTestDataset(submit, transform=val_transform)
test_loader = DataLoader(test_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=4)

models = []
for fold in range(CFG['n_folds']):
    model = UltimateFusionNet(CFG['model_name']).to(device)
    model.load_state_dict(torch.load(f'best_model_fold{fold+1}.pth'))
    model.eval()
    models.append(model)

ensemble_probs = []
with torch.no_grad():
    for f, t in tqdm(test_loader, desc="Ensemble Inference"):
        f, t = f.to(device), t.to(device)

        batch_probs = 0
        for model in models:
            logits1 = model(f, t)
            prob1 = torch.sigmoid(logits1)

            # TTA (좌우반전)
            f_flip = torch.flip(f, dims=[3])
            t_flip = torch.flip(t, dims=[3])
            logits2 = model(f_flip, t_flip)
            prob2 = torch.sigmoid(logits2)

            batch_probs += ((prob1 + prob2) / 2)

        batch_probs /= CFG['n_folds']
        ensemble_probs.extend(batch_probs.cpu().numpy())

ensemble_probs = np.array(ensemble_probs)

# unstable이 1이었으므로, 예측값은 unstable_prob임
submit['unstable_prob'] = np.clip(ensemble_probs, 1e-9, 1 - 1e-9)
submit['stable_prob'] = 1.0 - submit['unstable_prob']

submit.to_csv('submission_final_SOTA_4ch_40epochs.csv', index=False)
print("🚀 최종 파일 생성 완료: submission_final_SOTA_4ch_40epochs.csv")

Ensemble Inference: 100%|██████████| 32/32 [01:19<00:00,  2.48s/it]

🚀 최종 파일 생성 완료: submission_final_SOTA_4ch_40epochs.csv


In [ ]:
import shutil
drive_folder = "/content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/dacon/가중치/5-fold-v4"

# 폴더가 없으면 생성
if not os.path.exists(drive_folder):
    os.makedirs(drive_folder)
    print(f"📂 새 폴더를 생성했습니다: {drive_folder}")

# 3. 파일 복사 (파일명에 실험 정보 추가)
source_file = "best_model_fold5.pth"
destination_file = os.path.join(drive_folder, "best_model_fold5.pth")


if os.path.exists(source_file):
    shutil.copy2(source_file, destination_file)
    print(f"✅ 복사 완료: {destination_file}")
else:
    print("❌ 복사할 파일(best_model.pth)을 찾을 수 없습니다.")

✅ 복사 완료: /content/drive/MyDrive/랭체인 AI 영상객체탐지분석 플랫폼 구축/dacon/가중치/5-fold-v4/best_model_fold5.pth
